<a id="chapter-02"></a>

# 第 02 章　PyTorch 与计算机视觉

[上一章](../01%20%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C%E4%B8%8E%E5%8F%8D%E5%90%91%E4%BC%A0%E6%92%AD/notes.ipynb) | [学习路径](../README.md) | [下一章](../03%20Transformer%E4%B8%8E%E5%A4%A7%E6%A8%A1%E5%9E%8B%E5%85%A5%E9%97%A8/notes.ipynb)

> **本章主线**：本章聚焦**张量与训练流程、图像输入、分类、分割和去噪**。完成后进入第 03 章学习序列与注意力。

**章节目录**

- [2.1 PyTorch 与前馈神经网络](#part-02-01)
- [2.2 OpenCV 与图像输入](#part-02-02)
- [2.3 卷积神经网络](#part-02-03)
- [2.4 U-Net 与语义分割](#part-02-04)
- [2.5 图像去噪](#part-02-05)

**阅读层级**：章标题 → `章.节` → `章.节.小节` → `章.节.小节.知识点`。每节由分隔线与学习目标开始，正文细节使用加粗标签。

**运行约定**：以当前章节为工作目录，先执行公共导入与输出路径配置，再执行目标实验的导入、数据准备和模型定义。同名变量可能在不同实验中重定义；完整训练按需运行。

**运行路线**：章首初始化 → 2.1 MNIST → 2.2 图像输入 → 2.3 ImageCNN → 2.4 U-Net → 2.5 去噪。没有 DRIVE 时可跳过 2.4.2 的真实数据训练，但先执行 2.4.1 的结构演示。完整训练单元格默认只跑 1 epoch，按需增加。


In [ ]:
from pathlib import Path
# 从仓库根目录或本章目录启动内核均可；不静默切换工作目录。
ROOT = Path.cwd().resolve()
if ROOT.name.startswith('02 '):
    ROOT = ROOT.parent
CHAPTER_DIR = next(ROOT.glob('02 *'), None)
if CHAPTER_DIR is None or not (ROOT / 'AGENTS.md').is_file():
    raise RuntimeError('请从仓库根目录或第 02 章目录启动 Jupyter。')
OUTPUT_DIR = ROOT / 'outputs/chapter-02'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = CHAPTER_DIR / 'data'
IMAGE_DIR = CHAPTER_DIR / 'images'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as T
from PIL import Image

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
%matplotlib inline


**运行前检查**：先执行上面两格。模型权重与图片都写入 `OUTPUT_DIR`；保存模型使用后文的 `state_dict` 示例，不再引入无人调用的 pickle 包装函数。

本章只定义一次 `FNN`、`ImageCNN` 和 `UNet`。2.4 复用 2.3 的图像网络，2.5 也复用该网络；按节跳读时先执行对应定义单元格。数据变量分别使用 `mnist_*`、`segmentation_*`、`denoising_*`，避免互相覆盖。


---

<a id="part-02-01"></a>

## 2.1 PyTorch 与前馈神经网络

> **本节目标**：将张量、网络、损失与优化器连接成 MNIST 分类流程。

**小节导航**

- [2.1.1 前馈神经网络 FNN Feedforward Neural Networks](#sub-02-01-01)
- [2.1.2 前馈神经网络实例 MNIST Classification](#sub-02-01-02)

[返回章节目录](#chapter-02)

**输入 → 输出**：`[B,1,28,28]` 图像 → `[B,10]` 对数概率。先读张量、数据与模型定义，再看训练、测试和 checkpoint。

**练习检查点**：改变 batch size 后核对形状；确认 `exp(log_probs)` 每行和为 1；比较训练模式与评估模式的职责。


<a id="sub-02-01-01"></a>

### 2.1.1 前馈神经网络 FNN Feedforward Neural Networks


<a id="sub-02-01-01-01"></a>

#### 2.1.1.1 前馈神经网络的定义 Definition


A feed-forward neural network $f(\cdot , \theta)$ is a **directed acyclic graph**, parametrized by $\theta$, that applies a series of transformation to an input $\mathbf{x} \in \mathbb{R}^d$, layer-wise, and without recursion, to
produce an output $y = f(\mathbf{x}, \theta) \in \mathbb{R}^s$ , as depicted in below.

<p align="center">
<img src="./images/layer_wise_fnn.png" width="550" title="Layer computation in a feed-forward neural network">
    
Given a $K$ layer network, the $k$-th layer is characterized by a function $f_k$ parametrized by $\theta_k$. In other words,     
    
$$f(\mathbf{x}, \theta) = f_K(\dots f_2(f_1(\mathbf{x}; \theta_1); \theta_2) \dots; \theta_K)$$
    
The $f_k$'s are of the form
$$f_k(\mathbf{z}_{k-1}; \theta_k) = \mathbf{z}_k = \sigma_k(\mathbf{q}_k) = \sigma_k(\mathbf{W}_k\mathbf{z}_{k-1} + \mathbf{b}_k)$$
    
where $\theta_k = \{\mathbf{W}_k, \mathbf{b}_k\}$, $\mathbf{W}_k \in \mathbb{R}^{h_k\times h_{k-1}}$ is known as the **weight matrix**, $\mathbf{b}_k \in \mathbb{R}^{h_k}$ is the **bias vector**, $\mathbf{z}_k \in \mathbb{R}^{h_k}$ is the output of the $k$-th layer, $h_k$ is the dimension (the number of neurons) of the $k$-th layer, $\sigma_k$ is an **activation operator** (Softmax couples coordinates, unlike point-wise ReLU) known as **activation function** of the layer, and $\mathbf{q}_k$ is the **pre-activation vector**.

Typical **activation functions** include: 
- **tanh**: $x \mapsto \frac{\exp(2x) -1 }{\exp(2x) + 1}$,
- **sigmoid**: $x \mapsto \frac{1}{\exp(-x) + 1}$,
- **ReLU**:  $x \mapsto \max(0,x)$ (The Rectified Linear Unit ),
- **SoftMax**:  $\mathbf{x} \in \mathbb{R}^d  \mapsto [\dots, \frac{\exp(\mathbf{x}[i])}{\sum_{j=1}^d \exp(\mathbf{x}[j])}, \dots]$, <span style="color: red;">**常用于多元分类，可以视为“概率”**</span>,
- **LogSoftMax**:  $\mathbf{x} \in \mathbb{R}^d  \mapsto [\dots, \log{\frac{\exp(\mathbf{x}[i])}{\sum_{j=1}^d \exp(\mathbf{x}[j])}}, \dots]$, <span style="color: red;">**常用于多元分类，可以视为“log概率”，与 NLLLoss 联用**</span>。

Given an output $\mathbf{z} \in \mathbb{R}^s$ and a ground-truth $\mathbf{y} \in \mathbb{R}^s$, commonly used **loss functions** are:
- **MSE**: $(\mathbf{z}, \mathbf{y}) \mapsto \frac{1}{s}\|\mathbf{z} - \mathbf{y}\|_2^2$, (Mean Squared Error),
- **MAE**: $(\mathbf{z}, \mathbf{y}) \mapsto \frac{1}{s}\|\mathbf{z} - \mathbf{y}\|_1$, (Mean Absolute Error),
- **CE**: $(\mathbf{z}, \mathbf{y}) \mapsto - \langle \mathbf{y}, \log\mathbf{z} \rangle = - \sum_i \mathbf{y}[i] \log \mathbf{z}[i] $, (Cross-entropy),
- **NLL**: $(\mathbf{z}, i) \in \mathbb{R}^d * \mathbb{R} \mapsto -\mathbf{z[i]}$, (Negative-loglikelihood)

note, a **softmax** function should not be used in the final layer of the network when using **CrossEntropy**：
- 效率：CrossEntropyLoss 接收 logits，等价于 LogSoftmax 与 NLLLoss 的组合。先做 Softmax 会把概率再次当作 logits，改变损失和梯度，并非仅仅降低效率。

- 数值稳定性：另一个原因是数值稳定性。softmax 函数会将输出转换为概率，这可能会导致数值范围的丧失（因为概率的范围是0到1）。这可能会导致数值稳定性问题，特别是在计算损失的对数部分时。而 PyTorch 的 CrossEntropyLoss 在内部处理了这个问题，可以确保数值的稳定性。

因此，如果你使用的损失函数是 CrossEntropyLoss，你应该让网络的最后一层直接输出原始的、未归一化的分数（ 即 logits ），而不是通过 softmax 函数转换的概率。   
    
    
**Illustration of a FNN**
<p align="center">
<img src="./images/fnn_example.png" width="450" title="Graphical view of a FNN">
</p>



<a id="sub-02-01-01-02"></a>

#### 2.1.1.2 逆误差传播算法 Back-propagation


Given a set of network parameter $\theta$, and a dataset $\mathcal{D} = \{\mathbf{x}_i \in \mathbb{R}^d, \mathbf{y}_i \in \mathbb{R}^s\}$, the goal is to optimize the following function

$$
\underset{\theta}{ \text{ min } } \Big[ \ell(\theta) =  \frac{1}{n} \sum_{i=1}^n loss(f(\mathbf{x}_i; \theta), \mathbf{y}_i) \Big]
$$

Not only is the problem  in high dimension, it is also non-convex. Therefore, $\theta$ is updated iteratively based on first order information (the gradient) so as to reach a (local) minimum.

Let $\theta^t$ be the value of $\theta$ at the $t$-th iteration, $\theta^0$ being the initialization of the parameters of the network, the update rule of the standard gradient descent  is as follows,

$$
\theta^{t+1} \leftarrow \theta^{t} - \eta \nabla_{\theta^{t}}\ell(\theta^{t}) ,
$$

where $\nabla_{\theta} \ell(\theta)$ is the **gradient** of $\ell$ w.r.t to $\theta$, and $\eta$ is the 
step-size also known as **learning rate**. Back-propagation applies the chain rule from the output toward earlier layers to compute gradients; the optimizer updates parameters after this gradient computation, rather than updating weights layer by layer during back-propagation.

<p align="center">
<img src="./images/layer_wise_fnn_bprop.png" width="750" title="Back-propagation">
    
The **learning rate** is usually chosen experimentally based on the figure below. In practice, various different variants of gradient descent are used, and are built in functions into both pytorch and tensorflow. Some commonly used optimisers are:
- Stochastic gradient descent (SGD)
- Adadelta
- Adam
    
    <p align="center">
<img src="./images/learningrates.jpeg" width="350" title="Back-propagation">
        
*epoch* 是一个术语，指的是网络完成一次完整的遍历训练集并对参数进行更新的过程。


<a id="sub-02-01-01-03"></a>

#### 2.1.1.3 其他术语与标准训练流程 Terminologies & Standard Training Process


**张量 Tensor**

**"Tensor"** 是在深度学习中广泛使用的一种数据结构，它是矩阵和向量概念的一个高维扩展，在tensorFlow和PyTorch中是被<span style="color: red;">**作为tensor变量存储的一个多维数组，可以被索引**</span>。在形状上，tensor可以是0维（标量）、1维（向量）、2维（矩阵）、或者更高维度。比如，一张彩色图片可以被表示为一个3维tensor，其中三个维度分别对应图片的高度、宽度以及颜色通道（通常为红、绿、蓝三个颜色通道）。再如，一段视频可以被表示为一个4维tensor，四个维度分别对应时间、高度、宽度以及颜色通道。

特别的，一组数据也可以视为一个tensor，其第一个axis表示单个数据的下标，余下的axes表示该数据/tensor的各个维度。可以使用下面的命令创建一个简单的tensor，其维度为2，所有元素均为0。

In [ ]:
import torch

tensor_example = torch.zeros((2, 3)) # 创建一个2x3的张量，初始化为0

print(tensor_example)
print(tensor_example.size())         # 查询tensor的维度



**数据集的拆分与功能 Separate the data into three folds**

在机器学习和深度学习中，我们通常将数据集分为三个部分：**训练集（Training Set）**、**验证集（Validation Set）**和**测试集（Test Set）**。

**验证集**是用来在模型训练过程中**进行性能评估和调整模型超参数的数据集**。验证集在训练过程中起到一个“中间人”的角色。具体来说，我们使用训练集来训练模型，然后在验证集上评估模型的性能，并根据验证集的表现来调整模型的超参数（如学习率、正则化参数等）。这样做的目的是为了防止模型过拟合训练数据。如果我们只有训练集和测试集，那么在调整模型和超参数时，我们可能会**过度依赖测试集的表现，这可能导致模型过拟合测试集**。验证集提供了一种在训练过程中评估模型的方法，而不必依赖测试集。

最后，一旦我们的**模型参数和超参数调整到最佳状态**，我们会在**测试集**上进行最后的评估，这可以提供一个公正的、未见过的数据集对模型性能进行评估，这也更接近模型在实际环境中的表现。

<p align="center">
<img src="./images/datasep.png" width="350" title="Dataset separation." >
</p>


Assessing the model performance on the validation set

<p align="center">
<img src="./images/accuracies.jpeg" width="350" title="Dataset separation." >
</p>


<a id="sub-02-01-02"></a>

### 2.1.2 前馈神经网络实例 MNIST Classification

**MNIST**数据集是一个手写数字识别的数据集，包含了6万张28x28像素的训练图像和1万张测试图像，每张图像都是一个0-9的手写数字。We will have a dataset $ \mathcal{D} = \{ (x_i,y_i) \}$ of $n$ images $x_i$ and $n$ associated labels $y_i$, with $y_i = \{0, 1, \cdots, 9\}$, $i=1,\cdots,n$.

下面这段代码是使用PyTorch的一个子库torchvision来加载MNIST数据集。torchvision是一个用于计算机视觉的库，它包含了很多著名的数据集、模型结构和图像转换工具。


<a id="sub-02-01-02-01"></a>

#### 2.1.2.1 获取训练集和测试集



**接口分工**：`datasets.MNIST` 提供图像和数字标签，`T.ToTensor()` 将每张图转为 `[1,28,28]` 张量。训练集与测试集分别实例化。


In [ ]:
# 创建一个MNIST数据集的实例，它将作为训练数据。
mnist_train = datasets.MNIST(
    root = DATA_DIR, # 指定数据集的根目录。这意味着数据集将会被下载到这个目录中，或者如果数据集已经存在，它将在这个目录中被找到。
    train = True,  # 指定我们需要训练集。MNIST数据集包含训练集和测试集，通过设置这个参数为True，我们表明我们需要训练集。                      
    transform = T.ToTensor(), # 指定了一个转换，将图像转换为张量。这是一个常见的转换，因为神经网络通常使用tensor作为输入。
    download = True)        # 如果数据集不在root指定的目录中，通过设置这个参数为True，代码会自动下载数据集。

# 与上面类似，但这里创建了一个MNIST数据集的实例作为测试数据。
mnist_test = datasets.MNIST(
    root = DATA_DIR, 
    train = False, 
    transform = T.ToTensor() )


**T.ToTensor()** 可以把PIL-Image或者NumPy的ndarray转换为PyTorch的张量变量**(tensor)**，对于 uint8 数组及常见的 8 位 PIL 图像，会把像素值从[0, 255]缩放到[0.0, 1.0]；其他数据类型并不一定缩放。

In [ ]:
#print mnist_train
print(mnist_train, end='\n\n') # train_data的结构为一个可迭代对象，第i个元素为（ feature_data , label ）

#print mnist_test
print(mnist_test, end='\n\n')


In [ ]:
print(mnist_train.data.size(), end='\n\n')    # mnist_train.data 是一个包含了所有样本特征的tensor，第一个axis表示数据下标
print(mnist_train.targets.size(), end='\n\n') # mnist_train.targets 是一个包含了所有样本标签的tensor，第一个axis表示数据下标

# Plot one train data
plt.imshow(mnist_train.data[1000], cmap ='gray');



**利用 torch.utils.data.DataLoader 加载数据集**



通常训练FNN，在单次训练周期（epoch）内，我们并**不会直接将整个训练集一次性地**喂给模型，而是会将训练集切分成特定大小的批次，**以“小批量（batch）”的形式传递至模型中（细嚼慢咽）**，形成如下流程：<span style="color: red;">**for epoch : for batch : update parameter**</span>
    
下段先定义一个训练 DataLoader，正式实验再分别创建训练和测试 DataLoader。**torch.utils.data.DataLoader** 是 PyTorch 提供的一个实用工具，它可以被用来加载数据集。它接受一个总的数据集（这里是 mnist_train 和 mnist_test）和一些其他参数，返回一个可迭代的对象，用于在训练或测试模型时获取数据。    

In [ ]:
# Windows/Jupyter 默认单进程读取，避免额外子进程的启动开销。
loader = DataLoader(mnist_train, batch_size=100, shuffle=True, num_workers=0)


**loader** 可以视为数据集（ 此处为 mnist_train ）的一个由大小为batch_size的batch构成的**划分**，不论是否打乱，其大小/长度等于   
**ceil(data_size / batch_size)**（常规 map-style 数据集、默认采样器且 `drop_last=False`；`drop_last=True` 时取 floor），我们采用如下两种标准格式代码遍历loader，其中i为当前batch的下标，**features**, **labels**分别为当前batch下样本的<span style="color: red;">**样本特征张量**与**样本标签张量**，</span>他们的**第一个索引（axis）为该样本在当前batch中的下标**

In [ ]:
# 演示一批即可，不需要为了查看形状遍历全部训练集。
features, labels = next(iter(loader))
print('features:', features.shape, features.dtype)
print('labels:', labels.shape, labels.dtype)
# 完整训练时使用：for features, labels in loader: ...


<a id="sub-02-01-02-02"></a>

#### 2.1.2.2 构建网络


**Python 类 (Class) 的构建**

在Python中，你可以通过定义一个类来创建自定义的数据类型。类可以包含方法（函数）和属性（变量）。其基本格式为：
```python
class MyClass:
    
    # 初始化方法
    def __init__(self, **args_init):
        self.instance_prop_1 = ...  # 实例属性
        self.instance_prop_2 = ...  
    
    # 类属性 
    class_prop_1 = ...
    class_prop_2 = ...

    # 实例方法
    def func_1(self, **args1):
        ...
        return 
    
    def func_2(self, **args2):
        ...
        return  
```

在Class中，<span style="color: red;">**不必自行定义初始化方法；需要初始化实例状态时再定义 `__init__`**</span>，**self** 指代实例instance本身，在实例方法中可以用**self.instance_prop** 与 **self.class_prop** 调用属性。下面是特定属性和方法的在class外部的调用：
```python
instance = MyClass(**args_init) # 调用初始化方法创建对象
instance.class_prop_1           # 调用类属性 
instance.instance_prop_1        # 调用实例属性
instance.func_1(**args1)        # 调用实例方法
```


In [ ]:
# 简单的例子
class MyClass:
    # 类属性 
    greeting = "Hello, World!"

    # 初始化方法
    def __init__(self, name):
        self.name = name  # 实例属性

    # 实例方法
    def greet(self):
        return f"{self.greeting}, {self.name}!"

instance = MyClass(name = 'Tom')

print(instance)
print(instance.greeting)
print(instance.name)
print(instance.greet())



**神经网络的构建**



**从普通类到网络模块**：下面继承 `nn.Module` 并调用 `super().__init__()`，让 PyTorch 注册参数和子模块；输入数据通过 `forward` 描述的运算流动。


In [ ]:
class FNN(nn.Module): # 这一行定义了一个新的神经网络类，继承了PyTorch的基础模块类nn.Module，生成对象适用于其所有方法属性
    
    def __init__(self):
        
        # 因为继承自nn.Module，这行代码调用了基类 nn.Module 的初始化函数
        # super()被用来调用父类（也就是nn.Module）的方法
        super().__init__()

        # 定义一个全连接层 1，输入的特征数量是28*28，输出的特征数量是 512
        # 输入的特征数量是28*28是因为我们要处理 28x28 像素的图像，并且我们会把图像拉平成一个一维向量
        self.hidden1 = nn.Linear(28*28, 512) 
        self.hidden2 = nn.Linear(512, 10)

        self.relu = nn.ReLU()
        self.logsoftmax = nn.LogSoftmax(dim=1)
        
    def forward(self, x):          # x 为 features (of the whole batch)
        
        # 将 x 重塑成一个大小为[batch_size, 28*28]的二维张量，
        # 第一维是当前batch中的下标，第二维是 -1 表示自动计算其他所有feature的堆叠
        x = x.flatten(start_dim=1)  
        x = self.hidden1(x)        # 相当于调用 nn.Linear(28*28, 512) 对象，输出为[batch_size, 512]的二维张量
        x = self.relu(x)           # 相当于调用 nn.ReLU() 对象，输出为[batch_size, 512]的二维张量
        x = self.hidden2(x)        # 相当于调用 nn.Linear(512, 10) 对象，输出为[batch_size, 10]的二维张量
        x = self.logsoftmax(x)     # 相当于调用 nn.LogSoftmax(dim=1) 对象，对各行进行标准化使行和为1后集体取对数
        
        return x


上述代码中有几项要点：
- <span style="color: red;">**nn.Linear(inputNodesNum, outputNodesNum)**</span> 是 PyTorch 中的一个模块，用于应用线性变换到输入数据，形式为 y = xA^T + b，其中 A 是模块的权重，b 是偏置项。当你创建一个 nn.Linear 对象时，你需要提供输入特征的数量和输出特征的数量。然而，你**不需要显式地提供权重矩阵 A 和偏置向量 b**。这是因为这些值在创建 nn.Linear 对象时会被自动初始化，并在训练过程中通过反向传播和优化器进行更新。     
 
 
- 假设我们有一个形状为 <span style="color: red;">**[batch_size, inputNodesNum]**</span> 的 tensor x 和一个 nn.Linear 对象 linear_layer。我们可以通过下面的方式调用这个对象：**output = linear_layer(x)** 这里，output 是应用了线性变换的结果，其形状为 <span style="color: red;">**[batch_size, outputNodesNum]**</span>。


- <span style="color: red;">**nn.Module**</span> 是 PyTorch 的一个基础类，<span style="color: red;">**所有的神经网络模块都应该继承这个类**</span>。它有一些重要的属性和方法，对于建立神经网络非常有用：
 - **parameters()**: 方法返回模块的所有参数（nn.Parameter 对象），可以把模型的所有参数传递给优化器 optimizer。
 - **named_parameters()**: 方法除了返回参数对象外，还会返回每个参数的名称。
 - **zero_grad()**: 方法清除模块参数的梯度；当前默认 `set_to_none=True` 会将梯度设为 None。这通常在反向传播前被使用。
 - **to(device)**: 方法把模块的所有参数移动到指定的设备。这对于 GPU 计算很有用。
 - **state_dict()**: 方法返回包含模块所有状态信息（参数和缓存）的字典。
 

- 自定义的神经网络大类里必须有一个根据实际情况所写的 <span style="color: red;">**forward(self, x)**</span> 以覆盖 nn.Module 类中的 forward 方法，在调用时 应使用 **model(x)**，它会处理模块 hooks 等机制；直接调用 **model.forward(x)** 会绕过这些机制。

In [ ]:
fnn_model = FNN()
for name, param in fnn_model.named_parameters():
    print(name, tuple(param.shape), 'trainable:', param.requires_grad)



<a id="sub-02-01-02-03"></a>

#### 2.1.2.3 训练网络

**可以直接从这里开始运行**

**训练准备**：复用已经检查过的 MNIST 和 FNN。设备由章首的 `device` 确定，模型、特征与标签必须在同一设备；下面的训练函数负责迁移。


In [ ]:
# 复用 2.1.2 已加载的 MNIST，不重复下载或初始化。
mnist_loaders = {
    'train': DataLoader(mnist_train, batch_size=100, shuffle=True, num_workers=0),
    'test': DataLoader(mnist_test, batch_size=100, shuffle=False, num_workers=0),
}


In [ ]:
# 复用上面的 FNN；输出是 log-probabilities，应与 NLLLoss 配对。
fnn_model = FNN()
with torch.no_grad():
    log_probs = fnn_model(torch.zeros(2, 1, 28, 28))
assert log_probs.shape == (2, 10)
torch.testing.assert_close(log_probs.exp().sum(dim=1), torch.ones(2))


**x.view(x.size(0), -1)** 表示将 x 的形状调整为一个二维张量，其中第一维度（即行数）保持不变，等于 x 的第一维度的大小（x.size(0)），第二维度（即列数）则自动计算以使得整个张量的元素总数保持不变。这常常用于在将多维度的特征输入全连接层（Fully Connected Layer）前将特征展平（Flatten）。

In [ ]:
def train_classifier(model, loaders, num_epochs=1, learning_rate=0.001, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(num_epochs):
        model.train()
        total_loss, count = 0.0, 0
        for features, labels in loaders['train']:
            features, labels = features.to(device), labels.to(device)
            # model(x) 经过 Module.__call__，保留 hooks 的行为。
            loss = F.nll_loss(model(features), labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.numel()
            count += labels.numel()
        if not count:
            raise ValueError('Empty training loader')
        print(f'Epoch {epoch + 1}: mean loss = {total_loss / count:.4f}')


上述 **##** 标出了常规训练步骤：先前向计算，再反向传播，最后更新参数；清梯度可放在前向之前或反向之前，梯度累积时另行安排。其中：
- **model.train()**：将模型设置为训练模式。这是因为有些模块在训练和评估时的行为是不同的。调用 .train() 方法会将这些模块设置为训练模式。
- **output = model(features) & loss = loss_func(output, labels)**：每一次的正传流程必须在逆向传播之前
- **optimizer.zero_grad()**：在进行反向传播之前，先将模型的所有参数的梯度清零。这是因为 PyTorch 在默认情况下会累积梯度，如果不清零的话，每次调用 .backward() 方法时，新计算的梯度会被加到之前的梯度上。
- **loss.backward()**：进行反向传播，计算出每个参数的梯度。
- **optimizer.step()**：根据计算出的梯度来更新模型的参数。

In [ ]:
def evaluate_classifier(model, loader, device='cpu'):
    model.to(device).eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            log_probs = model(features)
            total_loss += F.nll_loss(log_probs, labels, reduction='sum').item()
            correct += (log_probs.argmax(dim=1) == labels).sum().item()
            count += labels.numel()
    if not count:
        raise ValueError('Empty evaluation loader')
    return {'loss': total_loss / count, 'accuracy': correct / count, 'samples': count}


**with torch.no_grad()**：这是一个上下文管理器，用于暂时关闭自动求导。在评估模型时，我们不需要计算梯度，因此可以关闭自动求导以节省内存并加速计算。

在PyTorch中，**.item()** 是一个在单元素张量上调用的方法，用于将该单元素张量的值作为一个Python数字返回。当你有一个只包含一个元素的张量（比如在计算损失函数后的结果）并且你需要将其值作为一个普通的Python数字使用时，你会用到这个函数。

**.data** 可属性被用来获取纯数据的 Tensor，它包含的就只有数据，没有 grad_fn 和 grad 这些信息。这就意味着对 tensor.data 的修改不会影响梯度的计算和反向传播。如果你在某些操作中不想被追踪梯度信息，可以使用 .data 。在新版的 PyTorch 中，更推荐使用 with torch.no_grad(): 进行上下文管理，这样在该上下文范围内的操作都不会被追踪梯度。

注意对 tensor 类型的运算大多会<span style="color: red;">**自动累积梯度**</span>，影响梯度的计算和反向传播，在进行数值运算时应采用下列手法避免此类现象发生：
- 在 **with torch.no_grad()** 环境下运算
- 用 **.item()** 将该单元素张量的值作为一个纯数字返回
- 用 **.data** 将张量的值作为一个纯矢量返回

**torch.nn.functional.nll_loss(input=output, target=labels, size_average=False)**
- **input (Tensor)**: 是一个包含每个类别预测分数（通常是 softmax 函数的输出之前的分数）的张量。其形状为 (batch_size, num_classes) 或者更高维（但最后一维必须是类别数量 num_classes）。
- **target (Tensor)**: 是一个包含每个样本的真实类别标签的张量。其形状应当是 input 的形状去掉最后一维（对于形状为 (batch_size, num_classes) 的 input，target 的形状应为 (batch_size,)）。它应当包含每个样本的类别标签，为类别索引（整数）。
- **size_average (布尔值, 可选)**: 已弃用。指定是否对每个小批量的损失求平均。默认值为 True。False 为求 sum loss。
- **reduction (字符串, 可选)**: 代替size_average，指定如何减少损失：'none' | 'mean' | 'sum'。'none'：不减少；'mean'：输出的总损失除以输出元素的数量；'sum'：输出的总损失。默认值为 'mean'。
- 其他类似的方法还有 **F.cross_entropy, F.nll_loss, F.mse_loss等**

In [ ]:
# 先跑一个 epoch 检查流程；增加 epoch 前划分验证集，不用测试集调参。
fnn_model = FNN()
train_classifier(fnn_model, mnist_loaders, num_epochs=1, learning_rate=0.001, device=device)
print(evaluate_classifier(fnn_model, mnist_loaders['test'], device=device))


In [ ]:
fnn_model.eval()
# Plot multiple
figure = plt.figure(figsize=(15, 12))
cols, rows = 5, 5
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(mnist_test), size=(1,)).item()
    img, label = mnist_test[sample_idx]
    with torch.no_grad():
        output = fnn_model(img.unsqueeze(0).to(next(fnn_model.parameters()).device))
    pred_label = output.argmax(dim=1).cpu().numpy() # .numpy()将Tensor对象转换为NumPy ndarray对象
    figure.add_subplot(rows, cols, i)
    plt.title('gt = ' + str(label) + ', pred = ' + str(pred_label))
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()



<a id="sub-02-01-02-04"></a>

#### 2.1.2.4 模型的保存与读取



In [ ]:
# state_dict 保存参数；加载时必须构建相同架构。
torch.save(fnn_model.state_dict(), OUTPUT_DIR / 'fnn.pth')
fnn_model = FNN()
fnn_model.load_state_dict(torch.load(OUTPUT_DIR / 'fnn.pth', map_location='cpu', weights_only=True))
fnn_model.eval()


---

<a id="part-02-02"></a>

## 2.2 OpenCV 与图像输入

> **本节目标**：掌握颜色通道、图像读写与视频帧输入，为卷积网络准备数据。

**小节导航**

- [2.2.1 计算机眼中的图像：计算机视觉](#sub-02-02-01)
- [2.2.2 OpenCV库 (cv2)](#sub-02-02-02)

[返回章节目录](#chapter-02)

**输入 → 输出**：文件 → HWC/BGR 数组 → RGB 显示或灰度输出。默认内嵌显示，独立窗口需显式开启 `SHOW_WINDOWS`。

**衔接与练习**：对比同一张图片在 OpenCV 与 Matplotlib 中的通道顺序，保存灰度图后检查输出路径；不要把 HWC 直接当成模型所需的 BCHW。

示例图像：[Lenna.png](images/Lenna.png)。


**输入材料**：[视频样例 Ocean.mp4](data/opencv/Ocean.mp4)。图像示例使用本章 `images/` 中的配图。



<a id="sub-02-02-01"></a>

### 2.2.1 计算机眼中的图像：计算机视觉



**Def 2.2.1 RGB颜色通道与RGB值** 一张**彩色图片**通常是由很多微小的彩色像素 **(pixels)** 组成的，这些像素的颜色又由一个三元数组 **(R,G,B)** 决定。其中R,G,B分别代表红，绿，蓝的三原色通道 **(channels)**；数组 **(R,G,B)** 的值代表在对应颜色通道上的亮度，取值范围 **0 ~ 255**，越大的值代表**越高的亮度**。一般我们用**三张同样大小的矩阵**来代表一张彩色图片，如下图所示；特别的，一张黑白图片**(灰度图)**只有一条颜色通道。

<div align="center">
<img src=https://pic4.zhimg.com/80/v2-94eda292fdc88e4aa34b351d4b1be963_720w.webp width=60% height=60%/>
</div>


<a id="sub-02-02-02"></a>

### 2.2.2 OpenCV库 (cv2)



OpenCV的全称是 **Open Source Computer Vision Library**，是一个跨平台的计算机视觉库，可以在商业和研究领域中免费使用。OpenCV可用于开发实时的图像处理、计算机视觉以及模式识别程序。在Python中使用OpenCV，首先导入**cv2**库。

In [ ]:
import cv2



<a id="sub-02-02-02-01"></a>

#### 2.2.2.1 图像读取与输出



In [ ]:
# 读取图片
img = cv2.imread(str(IMAGE_DIR / 'Lenna.png')) # 以3DArray的形式返回图片的BGR矩阵
if img is None:
    raise FileNotFoundError(IMAGE_DIR / 'Lenna.png')
B = img[:,:,0] # B
G = img[:,:,1] # G
R = img[:,:,2] # R 

# 等价操作
B, G, R = cv2.split(img)

# 逆向操作
img_merge = cv2.merge((B,G,R)) # 注意(B,G,R)外有括号

# 生成图像副本
img_copy = img.copy() # 实际上是numpy中array的复制


注意此处，**cv2.imread(filename, flags=None)** 返回的是 <span style="color: red;">**(B,G,R)**</span> 而不是 **(R,G,B)**，但  **matplotlib** 里面是 **(R,G,B)**。此外，我们还可以通过参数 **flags** 设定读入图片的格式：   
**cv2.IMREAD_COLOR**    ：读入一副彩色图像。图像的透明度会被忽略，这是默认参数    
**cv2.IMREAD_GRAYSCALE**：以灰度模式读入图像，此时仅返回一个矩阵   
**cv2.IMREAD_UNCHANGED**：保留读取图片原有的颜色通道     

In [ ]:
# Notebook 默认内嵌显示；只有需要独立桌面窗口时开启。
SHOW_WINDOWS = False

def cv_show(name, img, wait_time=5000):
    if img is None:
        raise ValueError('Image could not be read')
    try:
        cv2.imshow(name, img)
        return cv2.waitKey(wait_time)
    finally:
        cv2.destroyAllWindows()

if SHOW_WINDOWS:
    cv_show('Lenna', img)


**cv2.imshow(name, img)**   中的参数二**img**为图像对象，类型是numpy中的ndarray类型   
**cv2.waitKey(wait_time)**  中的参数取 **wait_time = 0** 表示按任意键关闭窗口，不主动关闭，它必须接在 **cv2.imshow** 后面
**cv2.destroyAllWindows()** 指定关闭窗口的操作必须接在 **cv2.waitKey** 后面  
     
实际上，**cv2.waitKey(wait_time)** 的本质是表示<span style="color: red;">**暂停程序，在wait_time期间等待一个按键输入**</span>，如果超出预期时间，则跳过该行代码继续运行；**wait_time = 0** 实际表示**无限期暂停程序直至接收到按键输入**。同时，**cv2.waitKey(wait_time)** 也有返回值，如果在规定时间内接收到按键输入，则返回值为输入按键的**ASCII码**，否则返回**-1**，见下方代码：

In [ ]:
# 复用同一个函数；无需重复定义 cv_show。
if SHOW_WINDOWS:
    key = cv_show('Lenna', img)
    print('key code:', key)  # 无按键通常为 -1；ESC 通常为 27。


在上述代码块图片展示期间，按ESC键退出，所以返回值为27（ESC键对应的ASCII码）。

当然，也可以使用 **matplotlib** 显示图片，采用 **plt.imshow(img_RGB)** 即可，但注意，**img_RGB** 是一个表示 <span style="color: red;">**(R,G,B)**</span> 的3DArray，而非 **cv2.imread(filename, flags=None)** 返回的 <span style="color: red;">**(B,G,R)**</span>，利用 **cv2.cvtColor(img_BGR, cv2.COLOR_BGR2RGB)** 可以实现二者的转换：

In [ ]:
plt.subplot(121) # 左图为没有经过转换的输出
plt.imshow(img);
plt.subplot(122) # 右图为经过转换的输出
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB));


In [ ]:
imgGray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
if not cv2.imwrite(str(OUTPUT_DIR / 'Lenna_Gray.png'), imgGray):
    raise OSError('Failed to save grayscale image')



<a id="sub-02-02-02-02"></a>

#### 2.2.2.2 视频的读取与输出



视频的实际上是有多张静态图片，即 **帧 (frame)** 组合而成，读取视频相当于读取一条图片序列

In [ ]:
video_path = DATA_DIR / 'opencv/Ocean.mp4'
videoCap = cv2.VideoCapture(str(video_path))
print('video opened:', videoCap.isOpened())
videoCap.release()  # 只检查打开状态，也要释放句柄。


**videoCap.isOpened()** 会返回一个布尔值，判断当前的摄像头是否初始化成功，如果摄像头初始化失败，我们可以使用 **videoCap.open()** 函数来打开摄像头。我们将读取视频的整个过程封装成如下函数

In [ ]:
def read_video(video_path, max_frames=120):
    if max_frames < 1:
        raise ValueError('max_frames must be positive')
    capture = cv2.VideoCapture(str(video_path))
    frames = []
    try:
        if not capture.isOpened():
            raise FileNotFoundError(video_path)
        while len(frames) < max_frames:
            ok, frame = capture.read()
            if not ok:
                break
            frames.append(frame)
    finally:
        capture.release()
    return frames


**videoCap = cv2.VideoCapture(video_path)** -- 初始化所读取的视频     
**videoCap.isOpened()** -- 返回一个布尔值，判断当前的视频是否初始化成功    
**videoCap.read()** -- 返回一个布尔值 **isOpen** 表示帧是否被正确读取,可以通过检查这个返回值来判断视频是否结束；一个被读取帧的BGRarray **frame**     
**videoCap.release()** -- 用于关闭视频文件或相机设备，常用于结束对视频的操作时

In [ ]:
def frames_show(name, frames, wait_time=100):
    try:
        for frame in frames:
            cv2.imshow(name, frame)
            if cv2.waitKey(wait_time) & 0xFF == 27:
                break
    finally:
        cv2.destroyAllWindows()


In [ ]:
frames = read_video(video_path, max_frames=120)
gray_frames = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
if gray_frames:
    plt.imshow(gray_frames[0], cmap='gray')
    plt.axis('off')
if SHOW_WINDOWS:
    frames_show('Ocean gray', gray_frames, wait_time=50)


本例最多缓存 120 帧，避免整段视频占满内存。处理长视频时应逐帧读取、处理和输出；若保存视频，路径也应在 OUTPUT_DIR 下。


---

<a id="part-02-03"></a>

## 2.3 卷积神经网络

> **本节目标**：理解卷积、池化和局部特征提取，并连接到 PyTorch 实现。

**小节导航**

- [2.3.1 卷积神经网络的定义 Definition](#sub-02-03-01)
- [2.3.2 卷积神经网络的 Python 构建](#sub-02-03-02)

[返回章节目录](#chapter-02)

**输入 → 输出**：`[B,C,H,W]` → 同尺寸图像。本节代码展示全卷积编码器/解码器的空间形状，不是另一个 MNIST 分类训练。

**练习检查点**：用 `plot=True` 核对两次下采样与两次上采样；比较卷积局部共享参数与 FNN 展平输入的区别。网络定义将在分割与去噪复用。


<a id="sub-02-03-01"></a>

### 2.3.1 卷积神经网络的定义 Definition

卷积神经网络是一种特殊的前馈神经网络 -- 首次由 Yann LeCun 在 [这篇文章](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf) 中介绍，它在网络处理过程中考虑了其输入（图像）的结构。在图像处理中，传统前馈神经网络往往会面临参数爆炸的问题，而卷积神经网络的出现很好的解决了这个问题。相较于传统FNN，CNN有以下的优点：
- **能处理高维度的输入** 例如，给定一个 $200 \times 200 \times 3$ 的RGB图像，输入层与第一个隐藏层之间的权重矩阵的维度为 $h_1$，则其大小为 $120000 \times h_1$。因此，不仅参数数量会爆炸，而且计算过程也会耗时长久。
- **能利用输入的拓扑结构** 如 $2D$ 或 $3D$ 图像。与其拥有一个大的权重矩阵，不如拥有寻找某些模式的小矩阵更好。
- **能构建对某些变化的不变性** 在图像分类任务中，对输入进行微小变换后，输出应保持不变。

卷积神经网络的基本结构大致分为三部分：**卷积层 Convolutional Layer (CONV)**，**池化层 Pooling Layer (POOL)**，**全连接层 Fully-Connected Layer (FC)**:
- **卷积层 Convolutional Layer**：用于从局部初步提取图片特征，包含以下几步
 - **Zero-Padding**: 目的是解决图像边缘特征的提取问题，即将原始图像像素矩阵的边缘各扩充一行/列，扩充部分的像素值均设置为 **0** 。
 - **卷积/特征过滤**: 选取若干 **卷积核 kernel / 特征过滤器 filter** （训练中学习到的局部模式，不限于水平或垂直边缘），在扩充后的像素矩阵上依照指定 **步长 stride** 平移卷积核进行点积运算已提取特征。When the stride is 1 then we move the filters one pixel at a time. When the stride is 2 (or uncommonly 3 or more, though this is rare in practice) then the filters jump 2 pixels at a time as we slide them around. This will produce smaller output volumes spatially. 经由卷积操作后输出的图像称为一张 **特征图 Feature Map**, 每层卷积网络输入/输出的特征图个数称为 **通道 channel（类似于 RGB channel，比如彩色图在第一层卷积层的输入为 RGB 三个 channel，灰度图输入为一个 channel）**。
 - **RELU layer**: Apply an elementwise activation function, such as the max(0,x) thresholding at zero. This leaves the size of the tensor unchanged.
 
<table>
    <tr>
        <td>
          <img src="./images/CONV_padding.png" width="400" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/CONV_kernel.png" width="1000" /> 
        </td>
    </tr>
</table>


- **池化层 Pooling Layer**: 用于进一步压缩图片信息，包含以下几步
 - **划分特征矩阵**：用相同大小的 **模式 pattern** 划分由卷积层输出的特征矩阵。
 - **向下采样 Down sampling/pooling**：从划分出来的网格中依次采样出最能代表该部分的值置于一个新的矩阵/容器 volume 中，最常见的方法是**取其中的最大值**，称为**最大池化**。
 - **扁平化层 Flatten Layer**：将同张图片目前所得的所有信息 **按序叠加 stack** 成一个一维tensor作为后续FC的输入。

<table>
    <tr>
        <td>
          <img src="./images/Pooling1.png" width="400" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/Pooling2.png" width="1000" /> 
        </td>
        <td>
          <img src="./images/Pooling3.png" width="400" /> 
        </td>
    </tr>
</table>

<p align="center">
<img src="./images/conv_pool_illust.png" width="400" title="Fully Convolutional Neural Network Example." >
</p>

- **全连接层 Fully-Connected Layer**: 用于正式分类 compute the class scores, where each of the numbers correspond to a class score/probability. As with ordinary Neural Networks and as the name implies, each neuron in this layer will be connected to all the numbers in the previous volume.

<p align="center">
<img src="./images/CNN_full.png" width="1000" title="Fully Convolutional Neural Network Example." >
</p>


以上便是一个最简单的CNN。可以看出CNN的前半部分基本是一个 **压缩图片信息 + 降维** 的过程，实践中可以在 **扁平化之前插入多个卷积层/池化层** 以实现更复杂的图片识别、分类。或者用另一个卷积层代替 **FC** 形成 **全卷积网络（即网络中没有全连接操作）**，如下右图所示，此类网络一般用于图像处理，输出的结果为一张处理过后的图像，在后半层卷积网络中常包含有与pooling相逆的向上采样 **up-sampling/pooling** 或反卷积 **up-convolution** 操作用于恢复空间分辨率；上采样本身不能找回已经丢失的信息。强烈建议读者前往[这个CNN EXPLAINER](https://poloclub.github.io/cnn-explainer/)进行简单的实操理解。CNN的<span style="color: red;">**训练目标是得到卷积核的权重以及FC的参数**</span>。

<table>
    <tr>
        <td>
          <img src="./images/CNN2.jpg" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/fcn.png" width="500" /> 
    </tr>
</table>

相关资料
- [Stanford's CS231n CNN Tutorial](https://cs231n.github.io/convolutional-networks/)
- [【数之道 08】走进"卷积神经网络"](https://www.bilibili.com/video/BV1R5411w715/?spm_id_from=333.337.search-card.all.click&vd_source=ff901644057cda4596a72384c16c4fb4)



<a id="sub-02-03-02"></a>

### 2.3.2 卷积神经网络的 Python 构建



**先检查结构**：本节前向演示使用零张量，不需要下载 MNIST 或开始训练。重点观察空间尺寸变化；2.4 和 2.5 会使用真实输入与损失。


In [ ]:
class ImageCNN(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        # Input to conv1 will be image of shape [batch_size,1,28,28] 
        # (1 for one channel, height and width are 28 for this example)
        
        self.conv1 = nn.Sequential(   
            nn.Conv2d(in_channels=in_channels,out_channels=10,kernel_size=(3,3),padding=1), # output of this conv is of shape [BS,10,28,28]
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=(2,2)) # output of this is [BS,10,14,14] BS stands for batch_size
        )
    
        self.conv2 = nn.Sequential( 
            nn.Conv2d(in_channels=10,out_channels=20,kernel_size=(3,3),padding=1), # output of this is [BS,20,14,14]
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2)) # output of this is [BS,20,7,7]
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=20,out_channels=30,kernel_size=(3,3),padding=1), # Output of this [BS,30,7,7]
            nn.ReLU(),
            nn.Conv2d(in_channels=30,out_channels=30,kernel_size=(3,3),padding=1), # Output of this [BS,30,7,7]
            nn.ReLU()
        )
        
        # Deconvolution & Unpooling, sometimes not necessary
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=30,out_channels=20,kernel_size=(3,3),padding=1), # Output of this [BS,20,7,7]
            nn.ReLU(),
            nn.Upsample(scale_factor=2))                                           # Output of this [BS,20,7*2,7*2]
        
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=20,out_channels=10,kernel_size=(3,3),padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2)) #[BS,10,28,28]
        
        self.conv6 = nn.Sequential(
            nn.Conv2d(in_channels=10,out_channels=out_channels,kernel_size=(1,1)),
            nn.Sigmoid()
        )     

    def forward(self, x, plot=False):

        if x.shape[-2] % 4 or x.shape[-1] % 4:
            raise ValueError('ImageCNN requires height and width divisible by 4')
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.conv4(x3)
        x5 = self.conv5(x4)
        x6 = self.conv6(x5)
        
        if plot:
            print('Input shape', x.shape)
            print('After layer 1', x1.shape)
            print('After layer 2', x2.shape)
            print('After layer 3', x3.shape)
            print('After layer 4', x4.shape)
            print('After layer 5', x5.shape)
            print('After layer 6', x6.shape)

        return x6


- <span style="color: red;">**nn.Sequential()**</span> 是PyTorch中的一个容器模块，用于便捷地组合其他模块（例如nn.Conv2d, nn.ReLU等）。当nn.Sequential实例作为一个网络并传入一个输入时，它会将输入依次传递给它所包含的各个模块，并将一个模块的输出作为下一个模块的输入。


- **Pytorch** requires inputs to a convolutional layer to be of shape <span style="color: red;">**(BatchSize,Channels,Height,Width)**</span>. When using **dataloaders** this is done **automatically**, and so during your training loop you won't need to worry about this. However, if defining your own function may error as img is of shape (Channels,Height,Width), and not (1,Channels,Height,Width) -- (as you are essentially passing a batch of 1 when predicting a single image.  
 - **<span style="color: red;">tensor.unsqueeze(dim)</span>** 方法用来在位置 dim 为张量增加一个新维度，新的维度的大小为1。
 - **<span style="color: red;">tensor.squeeze(dim)</span>**  方法用来消除在位置 dim 的大小为1的维度。特别地，tensor.squeeze() 默认消除所有大小为1的维度。


- A similar problem here is when plotting the image. **matplotlib** expects your image to be of shape **(Height,Width,Channels) (i.e. the channels to be the final dim)**. To transpose an image from shape (Channels,Height,Width) to (Height,Width,Channels), use
 - **<span style="color: red;">img = np.transpose(img,[1,2,0])，其中[1,2,0]为要交换的轴下标。</span>** 
 - **<span style="color: red;">img = img.permute(1, 2, 0)，其中[1,2,0]为要交换的轴下标。</span>**

In [ ]:
# 这是图像到图像的全卷积网络，不是输出 10 类分数的分类器。
image_cnn = ImageCNN(in_channels=1, out_channels=1)
image = torch.zeros(1, 1, 28, 28)
with torch.no_grad():
    reconstructed = image_cnn(image, plot=True)
assert reconstructed.shape == image.shape


---

<a id="part-02-04"></a>

## 2.4 U-Net 与语义分割

> **本节目标**：从分类转向像素级预测，理解上采样、跳跃连接与分割实验。

**小节导航**

- [2.4.1 U-Net](#sub-02-04-01)
- [2.4.2 图像处理实例 - 语义分割 Semantic Segmentation](#sub-02-04-02)

[返回章节目录](#chapter-02)

**输入 → 输出**：RGB 图像 → 单通道血管概率。先用合成张量检查 U-Net，再准备 DRIVE 图像和掩码，最后比较 ImageCNN 与 U-Net。

**练习检查点**：确认 mask 只有 0/1；解释最近邻插值和跳连尺寸要求；推理前切换 eval。训练默认 1 epoch 只检查流程，不代表收敛结果。


<a id="sub-02-04-01"></a>

### 2.4.1 U-Net


<a id="sub-02-04-01-01"></a>

#### 2.4.1.1 反卷积 Deconvolution

下文代码中将使用的 **nn.ConvTranspose2d** 是 PyTorch 中的一个模块，用于实现**二维**的转置卷积操作，也被称为**反卷积，去卷积去卷积（Deconvolution）或者上采样**操作。该操作可以将输入的特征图尺寸放大，通常被用在图像生成或者语义分割等任务中，帮助模型从小尺寸的特征图恢复到原始的大尺寸。它是一种特殊的正向卷积，先按照一定的比例通过补0来扩大输入图像的尺寸，接着旋转卷积核(Kernel)，再进行正向卷积。 反卷积的操作只是恢复了矩阵的尺寸大小，并不能恢复每个元素值。需要注意的是，虽然转置卷积操作可以增大特征图的尺寸，但它并**不是卷积操作的逆操作。**

此外，<span style="color: red;">**nn.ConvTranspose2d 中的参数也与之前的 nn.Conv2d**</span>[ 略有不同：](https://blog.51cto.com/u_11466419/5459142)
- 在本段简化图示中考虑 dilation=1；$p=0$ 表示不补边，padding 并没有普遍限定在 0 到 k-1
- 用“插零后普通卷积”解释转置卷积时，有效补边与转置卷积参数不同：dilation=1 的简化情形为 $k-1-p'$。实际输出尺寸以本节末尾公式为准

我们可以使用常见卷积实现转置卷积。这里我们用一个简单的例子来说明，输入层为$2 \times 2$，先进行padding为 $p=2\ (p'=0)$ 的零填充，再使用步长Stride为1的$3 \times 3$卷积核进行卷积操作则实现了上采样，上采样输出的大小为$4 \times 4$。原链接：https://www.zhihu.com/question/48279880/answer/1682194600
<p align="center">
<img src="./images/deconv_padding.webp" width="300" title="Image." >
</p>



- 在传统卷积中，我们的 **stride** ($s$) 表示卷积核移动的步长
- 在反卷积中，**stride** ($s'$) 表示表示往输入图片每两个像素点中间填充0，而填充的数量就是 $s' - 1$，核的移动步长 ($s$) 默认为1

下面是两个不同 stride 下的反卷积操作，分别为$s'=1,\ s'=2$

<table>
    <tr>
        <td>
          <img src="./images/deconv_s1.gif" width="250" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/deconv_s2.gif" width="250" /> 
    </tr>
</table>

综上，对于 nn.ConvTranspose2d 输出的高度（H_out），同宽度（W_out），可以用以下公式进行计算：

$$H_{out} = (H_{in} - 1) \times s' - 2p' + dilation\times(k - 1) + output padding + 1$$

上述的其他参数的含义详见[此处可视化](https://github.com/vdumoulin/conv_arithmetic/blob/master/README.md)，默认 dilation 与 output padding 取 1 & 0，即默认：

$$H_{out} = (H_{in} - 1)\times s' - 2p' + k $$



<a id="sub-02-04-01-02"></a>

#### 2.4.1.2 U-Net的构造

U-Net是比较早的使用 **全卷积** 网络进行语义分割的算法之一，论文中使用包含 **压缩路径** 和 **扩展路径** 的对称U形结构在当时非常具有创新性，且一定程度上影响了后面若干个分割网络的设计，该网络的名字也是取自其U形形状。

U-Net的实验是一个比较简单的ISBI cell tracking数据集，由于本身的任务比较简单，U-Net紧紧通过30张图片并辅以数据扩充策略便达到非常低的错误率，拿了当届比赛的冠军。论文源码已开源，可惜是基于MATLAB的Caffe版本。虽然已有各种开源工具的实现版本的U-Net算法陆续开源，但是它们绝大多数都刻意回避了U-Net论文中的细节，虽然这些细节现在看起来已无关紧要甚至已被淘汰，但是为了充分理解这个算法，笔者还是建议去阅读作者的源码，地址如下：https://lmb.informatik.uni-freiburg.de

U-Net的U形结构如下图所示。网络是一个经典的全卷积网络（即网络中没有全连接操作）。网络的输入是一张 $572 \times 572$ 的 **边缘经过镜像操作的单通道（ channel = 1 ）图片（input image tile）**，关于"镜像操作"会在后文进行详细分析，网络的左侧（红色框）是由 **卷积**，**归一化** 和 **Max Pooling** 构成的一系列降采样操作，论文中将这一部分叫做 **压缩路径（contracting path）**。压缩路径由4个block组成，每个block使用了3个有效卷积和1个 Max Pooling 降采样，每次**降采样之后 Feature Map 的大小除以2**，因此有了图中所示的 Feature Map 尺寸变化。最终得到了尺寸为的 $32 \times 32$ Feature Map。同时，在每个block的**第一层卷积后channel数乘2**。

网络的右侧部分（绿色框）在论文中叫做**扩展路径（expansive path）**。同样由4个block组成，每个block开始之前通过**反卷积将 Feature Map 的尺寸乘2，同时将其通道数减半（最后一层略有不同），和左侧对称的压缩路径的 Feature Map 合并**，由于左侧压缩路径和右侧扩展路径的 Feature Map 的尺寸不一样，U-Net是通过**将压缩路径的 Feature Map 裁剪到和扩展路径相同尺寸的 Feature Map 进行归一化的（即图1中左侧虚线部分）**。扩展路径的卷积操作依旧使用的是有效卷积操作。由于该任务是一个二分类任务，所以网络有两个输出 Feature Map。
 
<p align="center">
<img src="./images/unet.png" width="800" title="Image." >
</p>

**为什么要归一化？**
- 归一化操作针对同一批次内所有位于同一通道的像素点进行（即计算均值方差时会考虑该批次内的所有图片），能够在网络训练过程中，提高网络的训练稳定性和学习速度
- 引入两个可学习的参数，即缩放参数（gamma）和平移参数（beta）。这两个参数的形状与输入通道数相同。通过训练，网络能够学习到最优的gamma和beta，使得模型的表现最好
- 采用函数 **nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)** 对有C个通道的，维度为(H,W)的二维数据进行批量归一化
 - 调用时输入参数为一个维度为 **(N,C,H,W)** 的tensor，分别代表 **(batch_size, channel_number, height, width)**
 - **num_features (int)** – C，channel_number
 - [document](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)
 
 补充材料
 - [知乎](https://zhuanlan.zhihu.com/p/43927696)
 - [原Paper](https://arxiv.org/pdf/1505.04597.pdf)

**阅读 U-Net 代码**：双卷积提取特征，下采样扩大感受野，上采样恢复分辨率，跳跃连接拼接相同尺度的编码特征。先看三个 block，再看它们在 UNet 中的组合。


In [ ]:
# 蓝色箭头 - two steps conv / double-conv block
class conv_block(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        # 添加 padding 确保 crop 时维数一致，原论文中无 padding
        super().__init__()
        self.activation_fn = nn.ReLU()
        
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels,  kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_channels),
            self.activation_fn,
            
            nn.Conv2d(out_channels, out_channels,  kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_channels),
            self.activation_fn 
        )
    
    def forward(self, x):
        x = self.conv(x)
        return x

# Contracting Path - conv, conv, pool
class down_block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = conv_block(in_channels, out_channels)
        self.pool = nn.MaxPool2d((2,2))

    def forward(self, inputs):
        x = self.conv(inputs)
        p = self.pool(x)
        return x,p      # 注意由于在反卷积时会拼接先前的 Feature map，固要返回 x

# Expanding Path - up, conv, conv
class up_block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2, padding=0) ## See 3.1.1
        self.conv = conv_block(out_channels+out_channels, out_channels)
    
    def forward(self, inputs, skip):
        x = self.up(inputs)
        x = torch.cat([x, skip], dim=1) # 拼接相同空间尺寸的特征；此版本未执行裁剪
        x = self.conv(x)
        return x

# Construct the whole U-Net
class UNet(nn.Module):

    def __init__(self, in_channels=3, out_channels = 1, f=(64,128,256,512,1024)):
        super().__init__()

        self.encoder1 = down_block(in_channels, f[0])
        self.encoder2 = down_block(f[0],f[1])
        self.encoder3 = down_block(f[1],f[2])
        self.encoder4 = down_block(f[2],f[3])

        self.bottleneck = conv_block(f[3],f[4])

        self.decoder1 = up_block(f[4],f[3])
        self.decoder2 = up_block(f[3],f[2])
        self.decoder3 = up_block(f[2],f[1])
        self.decoder4 = up_block(f[1],f[0])

        self.outputs = nn.Conv2d(f[0], out_channels, kernel_size = 1, padding = 0)
        self.sigmoid_layer = nn.Sigmoid()

    def forward(self, inputs, plot=False):
        if inputs.shape[-2] % 16 or inputs.shape[-1] % 16:
            raise ValueError("UNet requires height and width divisible by 16")
        c1, p1 = self.encoder1(inputs)
        c2, p2 = self.encoder2(p1)
        c3, p3 = self.encoder3(p2)
        c4, p4 = self.encoder4(p3)

        bn = self.bottleneck(p4)

        d1 = self.decoder1(bn, c4)
        d2 = self.decoder2(d1, c3)
        d3 = self.decoder3(d2, c2)
        d4 = self.decoder4(d3, c1)

        outputs = self.outputs(d4)
        outputs = self.sigmoid_layer(outputs)
        
        if plot:
            print('Input shape', inputs.shape)
            print('After layer 1', p1.shape)
            print('After layer 2', p2.shape)
            print('After layer 3', p3.shape)
            print('After layer 4', p4.shape)
            print('After bn', bn.shape)
            print('After layer 6', d1.shape)
            print('After layer 7', d2.shape)
            print('After layer 8', d3.shape)
            print('After layer 9', d4.shape)
            print('Output shape', outputs.shape)

        return outputs
    
# 用小通道数检查尺寸，避免只做前向演示就占用大量内存。
unet_widths = (8, 16, 32, 64, 128)
unet_demo = UNet(f=unet_widths).eval()
with torch.no_grad():
    result = unet_demo(torch.zeros(1, 3, 32, 32), plot=True)
assert result.shape == (1, 1, 32, 32)


当前实现采用 padding 保持卷积前后尺寸，没有实现原论文的裁剪。四次 2× 下采样要求输入高、宽均为 16 的倍数；572 会产生 71 与 70 的跳连尺寸不匹配，因此代码在入口明确报错。处理其他尺寸需先统一 resize/pad，并在输出端对应还原，不能直接拼接。



<a id="sub-02-04-02"></a>

### 2.4.2 图像处理实例 - 语义分割 Semantic Segmentation

语义分割（Semantic Segmentation）是计算机视觉中的一个重要任务，其目标是理解图像在像素级别上的内容。这意味着该任务不仅要求模型识别图像中的物体，还要求模型能够准确地划分出每个物体的边界，即对图像中的每个像素进行分类。

在语义分割任务中，我们的目标是将图像划分为多个部分或区域，并对每个部分赋予相应的标签或类别。例如，在自动驾驶的应用中，我们可能需要对道路、人行道、行人、车辆、建筑物等不同类型的物体进行像素级别的识别和分割。

语义分割的主要挑战在于如何准确地划分物体的边界，以及如何处理类别间的不均衡（某些类别在图像中出现的频率远高于其他类别）。

- Image segmentation is the task of partitioning an image, or identifying an object in an image
- Particular value in medical imaging, highlighting objects of interest
- The target image is a binary image, where pixels = 0 are background and pixels = 1 are foreground. We want the network to output a binary image replicating this.

<table>
    <tr>
        <td>
          <span>DRIVE 原始配图未随笔记提供；请参见本章 README 的数据获取说明。</span>
        </td>
        <td>
        </td>
        <td>
          <span>DRIVE 原始配图未随笔记提供；请参见本章 README 的数据获取说明。</span> 
    </tr>
</table>

**真实数据实践（可选）**：以下单元格需要完整的 DRIVE 图像与人工掩码。暂时没有数据时，保留上面的结构检查结果，跳到 2.5；不要为继续执行伪造路径或标签。



<a id="sub-02-04-02-01"></a>

#### 2.4.2.1 读入数据 - 构建自己的 Data Loader



In [ ]:
def get_paths(base_path=DATA_DIR / 'DRIVE'):
    base_path = Path(base_path)
    splits = []
    for split, ids in [('training', range(21, 41)), ('test', range(1, 21))]:
        images = [base_path / split / 'images' / f'{i:02d}_{split}.tif' for i in ids]
        masks = [base_path / split / '1st_manual' / f'{i:02d}_manual1.gif' for i in ids]
        missing = [path for path in images + masks if not path.is_file()]
        if missing:
            raise FileNotFoundError(f'DRIVE image/manual mask missing: {missing[0]}; see chapter README')
        splits.append((images, masks))
    return tuple(splits)


**文件配对**：图像和人工掩码按同一编号配对，不能分别乱序排列。这里使用 `training/images/21_training.tif`、`test/images/01_test.tif` 及对应 `1st_manual/*_manual1.gif`。若拿到的数据版本没有测试掩码，应先准备明确标注的验证划分，不要把训练掩码当作测试真值。缺文件时入口会报告具体路径。

预处理统一在下面的 Dataset 中完成，不再保留未调用的 skimage 读取函数。


In [ ]:
class SegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, paths, size=(256, 256)):
        self.im_paths, self.gt_paths = paths
        if not self.im_paths or len(self.im_paths) != len(self.gt_paths):
            raise ValueError('Expected nonempty paired image and mask paths')
        self.size = size

    def __getitem__(self, index):
        # PIL 的 size 为 (width, height)，图像与标签使用相同目标尺寸。
        with Image.open(self.im_paths[index]) as source:
            image = source.convert('RGB').resize(self.size, Image.Resampling.BILINEAR)
            image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255
        with Image.open(self.gt_paths[index]) as source:
            mask = source.convert('L').resize(self.size, Image.Resampling.NEAREST)
            mask = torch.from_numpy(np.array(mask)).unsqueeze(0).gt(127).float()
        return image, mask

    def __len__(self):
        return len(self.im_paths)


In [ ]:
train_paths, test_paths = get_paths()
segmentation_train = SegmentationDataset(train_paths)
segmentation_test = SegmentationDataset(test_paths)
segmentation_loaders = {
    'train': DataLoader(segmentation_train, batch_size=4, shuffle=True, num_workers=0),
    'test': DataLoader(segmentation_test, batch_size=4, shuffle=False, num_workers=0),
}


**预处理与 Dataset 接口**

- 图像先转换为 RGB，再做双线性缩放，最终是 `[3,H,W]` 的 float32 张量，范围为 `[0,1]`。
- 二值掩码用最近邻缩放，最终为 `[1,H,W]` 的 0/1 浮点张量；不能与图像共用双线性插值，否则会改变离散标签边界。
- `T.Resize` 接收 PIL 图像或 Tensor，并非任意 NumPy 数组；`T.ToTensor` 的缩放行为取决于输入类型，不能理解为所有数据都会除以 255。参见 [Resize 官方接口](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Resize.html)。
- `__getitem__` 返回一个 `(image, mask)` 样本；`__len__` 返回样本数量。DataLoader 将样本拼成 batch，因此形状为 `[B,C,H,W]`。
- 若加入随机裁剪/翻转，图像和掩码必须使用同一组几何变换；此处仅做固定尺寸变换。


In [ ]:
images, masks = next(iter(segmentation_loaders['train']))
print('images:', images.shape, 'masks:', masks.shape)
assert images.shape[1] == 3 and masks.shape[1] == 1
assert set(masks.unique().tolist()) <= {0.0, 1.0}



<a id="sub-02-04-02-02"></a>

#### 2.4.2.2 构建神经网络



**基线复用**：使用 2.3 定义的 `ImageCNN(in_channels=3, out_channels=1)`，将 RGB 图像映射为单通道血管概率。没有跳跃连接，作为 U-Net 的对照。


**U-Net 复用**：使用 2.4.1 已定义的 `UNet`，不重复定义卷积块。下面的训练和加载都使用相同的 `unet_widths`，以保证 checkpoint 形状一致。


**输出与损失配对**：两个分割模型都以 Sigmoid 输出 `[0,1]` 概率，所以使用 BCELoss。若改为输出 logits，应同时改用 BCEWithLogitsLoss，不能只改其中一端。



<a id="sub-02-04-02-03"></a>

#### 2.4.2.3 训练网络



In [ ]:
def train_segmenter(model, loaders, num_epochs=1, learning_rate=0.001, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(num_epochs):
        model.train()
        total_loss, count = 0.0, 0
        for features, masks in loaders['train']:
            features, masks = features.to(device), masks.to(device)
            loss = F.binary_cross_entropy(model(features), masks)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * features.size(0)
            count += features.size(0)
        if not count:
            raise ValueError('Empty training loader')
        print(f'Epoch {epoch + 1}: mean BCE = {total_loss / count:.4f}')

def evaluate_segmentation(model, loader, device='cpu'):
    model.to(device).eval()
    intersection = predicted = target = count = 0
    with torch.no_grad():
        for features, masks in loader:
            prediction = model(features.to(device)) >= 0.5
            truth = masks.to(device) >= 0.5
            count += features.size(0)
            intersection += (prediction & truth).sum().item()
            predicted += prediction.sum().item()
            target += truth.sum().item()
    if not count:
        raise ValueError('Empty evaluation loader')
    # 整个 loader 上的像素级 micro Dice；不等于逐图 Dice 均值。
    return (2 * intersection / (predicted + target)) if predicted + target else 1.0


In [ ]:
segmentation_cnn = ImageCNN(in_channels=3, out_channels=1)
train_segmenter(segmentation_cnn, segmentation_loaders, num_epochs=1, device=device)


In [ ]:
segmentation_unet = UNet(in_channels=3, out_channels=1, f=unet_widths)
train_segmenter(segmentation_unet, segmentation_loaders, num_epochs=1, device=device)
print('CNN Dice:', evaluate_segmentation(segmentation_cnn, segmentation_loaders['test'], device))
print('U-Net Dice:', evaluate_segmentation(segmentation_unet, segmentation_loaders['test'], device))


In [ ]:
torch.save(segmentation_cnn.state_dict(), OUTPUT_DIR / 'cnn.pth')
torch.save(segmentation_unet.state_dict(), OUTPUT_DIR / 'unet.pth')



<a id="sub-02-04-02-04"></a>

#### 2.4.2.4 可视化



In [ ]:
# 本小通道 U-Net 的权重不能加载到默认 64 起始通道的架构。
segmentation_cnn = ImageCNN(in_channels=3, out_channels=1)
segmentation_unet = UNet(in_channels=3, out_channels=1, f=unet_widths)
segmentation_cnn.load_state_dict(torch.load(OUTPUT_DIR / 'cnn.pth', map_location='cpu', weights_only=True))
segmentation_unet.load_state_dict(torch.load(OUTPUT_DIR / 'unet.pth', map_location='cpu', weights_only=True))
segmentation_cnn.eval()
segmentation_unet.eval()


In [ ]:
dataset = segmentation_test
segmentation_cnn.eval()
segmentation_unet.eval()

with torch.no_grad():
    figure = plt.figure(figsize=(15, 8))
    rows = 3
    for i in range(0, rows):
        sample_idx = torch.randint(len(dataset), size=(1,)).item()
        img, label = dataset[sample_idx]

        CNN_output  = segmentation_cnn(img.unsqueeze(0).to(next(segmentation_cnn.parameters()).device)).cpu()
        Unet_output = segmentation_unet(img.unsqueeze(0).to(next(segmentation_unet.parameters()).device)).cpu()

        figure.add_subplot(rows, 4, 4*i + 1)
        plt.axis("off")
        plt.imshow(img.permute(1,2,0))
        plt.title("image")
        
        figure.add_subplot(rows, 4, 4*i + 2)
        plt.axis("off")
        plt.imshow(CNN_output.squeeze(), cmap="gray")
        plt.title("CNN probability")
        
        figure.add_subplot(rows, 4, 4*i + 3)
        plt.axis("off")
        plt.imshow(Unet_output.squeeze(), cmap="gray")
        plt.title("U-Net probability")
        
        figure.add_subplot(rows, 4, 4*i + 4)
        plt.axis("off")
        plt.imshow(torch.squeeze(label), cmap="gray")
        plt.title("ground truth (target label)")


---

<a id="part-02-05"></a>

## 2.5 图像去噪

> **本节目标**：学习含噪输入、重建目标和去噪结果的比较，完成图像学习主线。

**小节导航**

- [2.5.1 生成含噪图像](#sub-02-05-01)
- [2.5.2 无监督学习的去噪模型](#sub-02-05-02)
- [2.5.3 可视化](#sub-02-05-03)

[返回章节目录](#chapter-02)

**输入 → 输出**：带噪 MNIST → 去噪图像。复用 ImageCNN，仅用带噪观测的 TV 与保真项训练；干净图只用于展示和评估。

**练习检查点**：固定测试噪声后改变 λ，比较平滑程度和 MSE；一批样本的结果不代表完整测试集，更不能保证去噪优于输入。



<a id="sub-02-05-01"></a>

### 2.5.1 生成含噪图像



**从分割切换到去噪**：不再预测人工标注的血管掩码，而是约束输出接近观测图像且保持平滑。沿用 2.1 的 MNIST 输入，忽略数字类别标签。


In [ ]:
class NoisyImages(torch.utils.data.Dataset):
    def __init__(self, clean_dataset, sigma=70 / 255, seed=42):
        if sigma < 0:
            raise ValueError('sigma must be nonnegative')
        self.clean_dataset, self.sigma, self.seed = clean_dataset, sigma, seed

    def __len__(self):
        return len(self.clean_dataset)

    def __getitem__(self, index):
        clean, _ = self.clean_dataset[index]
        # 按样本编号固定噪声，重复评估可比较；只分配当前样本的 float32 张量。
        generator = torch.Generator().manual_seed(self.seed + index)
        noise = torch.randn(clean.shape, generator=generator, dtype=clean.dtype)
        noisy = (clean + self.sigma * noise).clamp(0, 1)
        return noisy, clean

denoising_train = NoisyImages(mnist_train, seed=42)
denoising_test = NoisyImages(mnist_test, seed=4242)
denoising_loaders = {
    'train': DataLoader(denoising_train, batch_size=100, shuffle=True, num_workers=0),
    'test': DataLoader(denoising_test, batch_size=100, shuffle=False, num_workers=0),
}


In [ ]:
noisy, clean = denoising_train[0]
fig, axes = plt.subplots(1, 2)
for ax, image, title in zip(axes, [clean, noisy], ['clean reference', 'noisy input']):
    ax.imshow(image.squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')


<a id="sub-02-05-02"></a>

### 2.5.2 无监督学习的去噪模型

- In our denoising setup we assume no label. Our dataset consists of only observed noisy images, i.e. $\mathcal{D} = \{ \mathbf{x}_i \in \mathbb{R}^{h \times w} \}_{i=1}^n$. This approach (not using labels) is known as **unsupervised learning**.

- We will implement a classic denoising model, whose loss function is composed of two terms:

$$ \mathcal{L}\ (\ \theta\ ) =   \textstyle \sum_{i=1}^{\ n} \left \{\  ||\  \nabla_{(x,y)}\ f(\mathbf{x}_i;\theta) \ || + \frac{\lambda}{2} ||\  f(\mathbf{x}_i; \theta) - \mathbf{x}_i \ ||^2\ \right \} $$

- where $\lambda>0$ is a parameter which we hand tune according to the strength of noise. If we have a large $\lambda$, the second term is more dominiant and our network output will be matched more closely to the input (i.e. $f(\mathbf{x}_i ; \theta) \approx \mathbf{x}_i$). If $\lambda$ is small, the first term will be more dominant and more smoothing will occur.


- 注意，一方面，$f$ 可以被视为关于输入图像的函数 $f(\mathbf{x}_i;\theta)$，另一方面也可以视为关于二维像素坐标 (x,y) 的函数 $f(x,y\ ;\ \mathbf{x}_i,\theta)$，所以损失函数中的梯度项是关于像素坐标的梯度而不是输入变量 $\mathbf{x}_i$ 的梯度.
**实现边界**：代码使用中心差分、复制边界和 epsilon 平滑的梯度模，再对 batch/通道/像素取平均，是 TV 正则化思想的教学离散实现。它不是 Noise2Noise 或 Noise2Void，也没有利用干净目标训练；训练损失下降不保证干净参考上的 MSE 同时下降。


In [ ]:
# This is a self-defined Loss function
def up_shift(f):
    g = torch.zeros_like(f)
    g[:, :, :-1, :] = f[:, :, 1:, :] # input f is of size (batch_size, C, H, W)
    g[:, :, -1, : ] = f[:, :, -1, :]
    return g

def down_shift(f):
    g = torch.zeros_like(f)
    g[:, :, 1:, :] = f[:, :, :-1, :]
    g[:, :, 0, : ] = f[:, :, 0, :]
    return g

def left_shift(f):
    g = torch.zeros_like(f)
    g[:, :, :, :-1] = f[:, :, :, 1:]
    g[:, :, :, -1 ] = f[:, :, :, -1]
    return g

def right_shift(f):
    g = torch.zeros_like(f)
    g[:, :, :, 1:] = f[:, :, :, :-1]
    g[:, :, :, 0 ] = f[:, :, :, 0]
    return g

def grad(f):
    f_x = (left_shift(f) - right_shift(f))/2
    f_y = (down_shift(f) - up_shift(f))/2 
    return torch.sqrt(f_x**2 + f_y**2 + 1e-7) # + 1e-7 防止 f_x, f_y 过小

class denoising_loss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, denoised, noisy, lambdaP):     
        TV_term = grad(denoised)
        Fit_Term = (lambdaP/2)*(denoised-noisy)**2
        loss = TV_term + Fit_Term
        return loss.mean()


**网络复用**：沿用 `ImageCNN(in_channels=1, out_channels=1)`。分割与去噪共享图像网络结构，但训练目标不同；这里不使用干净图像计算训练损失。


In [ ]:
def train_denoiser(model, loaders, num_epochs=1, lambdaP=8, learning_rate=0.001, device='cpu'):
    if lambdaP <= 0:
        raise ValueError('lambdaP must be positive')
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_func = denoising_loss()
    for epoch in range(num_epochs):
        model.train()
        total_loss, count = 0.0, 0
        for noisy, _ in loaders['train']:  # clean 只用于评估，不进入训练损失
            noisy = noisy.to(device)
            loss = loss_func(model(noisy), noisy, lambdaP)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * noisy.size(0)
            count += noisy.size(0)
        if not count:
            raise ValueError('Empty training loader')
        print(f'Epoch {epoch + 1}: mean TV/fidelity loss = {total_loss / count:.4f}')


In [ ]:
denoising_model = ImageCNN(in_channels=1, out_channels=1)
train_denoiser(denoising_model, denoising_loaders, num_epochs=1, lambdaP=10, device=device)


In [ ]:
torch.save(denoising_model.state_dict(), OUTPUT_DIR / 'denoising.pth')



<a id="sub-02-05-03"></a>

### 2.5.3 可视化



In [ ]:
denoising_model = ImageCNN(in_channels=1, out_channels=1)
denoising_model.load_state_dict(torch.load(OUTPUT_DIR / 'denoising.pth', map_location='cpu', weights_only=True))
denoising_model.eval()


In [ ]:
denoising_model.eval()
with torch.no_grad():
    noisy, clean = next(iter(denoising_loaders['test']))
    predicted = denoising_model(noisy.to(next(denoising_model.parameters()).device)).cpu()
    print('First batch noisy MSE:', F.mse_loss(noisy, clean).item())
    print('First batch denoised MSE:', F.mse_loss(predicted, clean).item())
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for row in range(3):
        for ax, image, title in zip(axes[row], [noisy[row], predicted[row], clean[row]], ['noisy', 'denoised', 'clean reference']):
            ax.imshow(image.squeeze(), cmap='gray', vmin=0, vmax=1)
            ax.set_title(title)
            ax.axis('off')
plt.tight_layout()


---

**第 02 章完成：从图像学习转向序列学习**

已学习前馈训练、图像输入、卷积、分割与去噪。下一章从 RNN/LSTM 的隐藏状态和门控开始，再过渡到注意力与 Transformer。

[进入第 03 章：序列建模](../03%20Transformer%E4%B8%8E%E5%A4%A7%E6%A8%A1%E5%9E%8B%E5%85%A5%E9%97%A8/notes.ipynb#part-03-01)
